### Imports

In [1]:
import os, sys
import yaml
import toml



# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import get_images_from_directory

from eval_segmentation import write_seg_metrics_csv, write_avg_metrics_yaml_from_csv




2024-09-25 15:43:53.519407: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-25 15:43:53.533991: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-25 15:43:53.538424: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-25 15:43:53.552283: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-25 15:43:54.814315: W tensorflow/compiler/tf2

[09/25/24 15:43:59] WARNING  Your inference package version 0.18.1 is out of date! Please upgrade to ]8;id=705791;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=86179;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.20.1 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

### Config and Datasets

In [2]:
id_tag = '0004'

# Load the configuration file
config_path = f'/workspace/home01/pipeline_configs/config_{id_tag}.toml'
config = toml.load(config_path)

# Load YAML file into a dictionary
dataset_name = config["dataset"]["name"]
dataset_yaml_file = config["dataset"]["yaml_file"]
with open(dataset_yaml_file, "r") as file:
    datasets = yaml.safe_load(file)

# get dataset
dataset = datasets[dataset_name]

## Evaluate data
### Load data


In [3]:
if "use_existing" in config["segmentation"]:
    segmentation_id = config["segmentation"]["use_existing"]
    if segmentation_id == "gt":
        segmentation_path = dataset["gt_seg_path"]
    else:
        segmentation_path = f"/workspace/home/pipeline_outputs/{dataset_name}/{segmentation_id}/seg"
else:
    segmentation_path = (
        f"/workspace/home/pipeline_outputs/{dataset_name}/{id_tag}/seg"
    )
    segmentation_id = id_tag

segmentation_gt_path = datasets[dataset_name]['gt_seg_path']

segmentation_masks = get_images_from_directory(dir_path=segmentation_path) # might be wrong: filter with label
segmentation_masks_gt = get_images_from_directory(dir_path=segmentation_gt_path)

segmentation_masks_hint_path = "/workspace/home/data/ISIC2019/hint_data/ISIC2018_Task1_Training_GroundTruth"
segmentation_masks_hint = get_images_from_directory(segmentation_masks_hint_path)

### Calculate metrics

In [5]:
# create csv path
base_dir = f"/workspace/home/pipeline_outputs/{dataset_name}/{segmentation_id}"
os.makedirs(base_dir, exist_ok=True)
csv_path = os.path.join(base_dir, 'seg_metrics.csv')

if not os.path.exists(csv_path):
    os.mknod(csv_path)

    # create csv file
    dilation_kernel_size = config["inpainting"]["dilation"]["kernel_size"]
    dilation_iterations = config["inpainting"]["dilation"]["iterations"]
    dilation_function_params = {'kernel_size': dilation_kernel_size, 'iterations': dilation_iterations}
    write_seg_metrics_csv(csv_path, segmentation_masks, segmentation_masks_gt, segmentation_masks_hint, dilation_function_params)

# create yaml file
yaml_path = os.path.join(base_dir, 'seg_metrics_avg.yaml')
if not os.path.exists(yaml_path):
    write_avg_metrics_yaml_from_csv(csv_path, yaml_path)

